# AutoML — Aprendizaje Automático Automatizado

## Notebook educativo — Usando librerías de AutoML

### Objetivo
Explorar las principales librerías de **AutoML** que automatizan la selección de modelos, ingeniería de features y optimización de hiperparámetros, comparando sus enfoques, ventajas y limitaciones.

### Contenido
1. **¿Qué es AutoML?** — Motivación, pipeline y taxonomía
2. **Auto-sklearn** — AutoML basado en scikit-learn con meta-learning
3. **TPOT** — Optimización de pipelines con programación genética
4. **FLAML** — AutoML rápido y ligero de Microsoft
5. **H2O AutoML** — Plataforma escalable con stacking automático
6. **Optuna** — Optimización bayesiana flexible (framework-agnostic)
7. **PyCaret** — AutoML low-code para prototipado rápido
8. **LazyPredict** — Benchmark rápido de múltiples modelos
9. **Comparación final** — Benchmark en múltiples datasets

### Requisitos
```bash
pip install auto-sklearn tpot flaml h2o optuna pycaret lazypredict scikit-learn xgboost lightgbm
```

> ⚠️ **Nota**: Algunas librerías pueden tardar en instalar. En Colab, ejecuta las celdas de instalación según necesites.

In [ ]:
# ============================================================
# Instalación de librerías (descomentar las que necesites)
# ============================================================

# En Google Colab, ejecutar estas líneas:
# !pip install -q flaml[automl] optuna pycaret lazypredict tpot auto-sklearn h2o
# !pip install -q xgboost lightgbm catboost

# Para este notebook, instalaremos las más ligeras primero
# Las más pesadas (auto-sklearn, h2o) se instalan en sus secciones

import subprocess
import sys

def instalar(paquete):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

# Instalar las librerías ligeras
for pkg in ["flaml[automl]", "optuna", "lazypredict", "tpot"]:
    try:
        instalar(pkg)
        print(f"  ✅ {pkg}")
    except Exception as e:
        print(f"  ⚠️  {pkg}: {str(e)[:80]}")

print("\n✅ Instalación completada")

In [ ]:
# ============================================================
# Imports y datasets comunes
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import (load_iris, load_wine, load_breast_cancer,
                                load_digits, fetch_california_housing,
                                make_classification, make_regression)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, mean_squared_error,
                              r2_score, classification_report)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               RandomForestRegressor, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)

# --- Preparar datasets para benchmarks ---
def preparar_datasets():
    """Prepara múltiples datasets para comparar AutoML."""
    datasets = {}
    
    # 1. Iris (clasificación fácil)
    iris = load_iris()
    datasets['Iris (3 clases)'] = {
        'X': iris.data, 'y': iris.target, 'tipo': 'clf',
        'n_samples': len(iris.target), 'n_features': iris.data.shape[1]
    }
    
    # 2. Breast Cancer (clasificación binaria)
    bc = load_breast_cancer()
    datasets['Breast Cancer'] = {
        'X': bc.data, 'y': bc.target, 'tipo': 'clf',
        'n_samples': len(bc.target), 'n_features': bc.data.shape[1]
    }
    
    # 3. Wine (clasificación multiclase)
    wine = load_wine()
    datasets['Wine (3 clases)'] = {
        'X': wine.data, 'y': wine.target, 'tipo': 'clf',
        'n_samples': len(wine.target), 'n_features': wine.data.shape[1]
    }
    
    # 4. Sintético complejo
    X_syn, y_syn = make_classification(n_samples=1000, n_features=20,
                                        n_informative=10, n_redundant=5,
                                        n_classes=4, random_state=42)
    datasets['Sintetico (4 clases)'] = {
        'X': X_syn, 'y': y_syn, 'tipo': 'clf',
        'n_samples': 1000, 'n_features': 20
    }
    
    # 5. California Housing (regresión)
    try:
        cal = fetch_california_housing()
        # Submuestra para velocidad
        idx = np.random.choice(len(cal.target), 2000, replace=False)
        datasets['California Housing'] = {
            'X': cal.data[idx], 'y': cal.target[idx], 'tipo': 'reg',
            'n_samples': 2000, 'n_features': cal.data.shape[1]
        }
    except:
        X_reg, y_reg = make_regression(n_samples=2000, n_features=10, noise=10, random_state=42)
        datasets['Regresion sintetica'] = {
            'X': X_reg, 'y': y_reg, 'tipo': 'reg',
            'n_samples': 2000, 'n_features': 10
        }
    
    return datasets

datasets = preparar_datasets()
print("=== Datasets preparados ===")
for nombre, info in datasets.items():
    print(f"  {nombre}: {info['n_samples']} samples, {info['n_features']} features, tipo={info['tipo']}")

print("\n✅ Listo para AutoML")

## 1. ¿Qué es AutoML?

### Definición
**AutoML** (Automated Machine Learning) es el proceso de automatizar las etapas del pipeline de ML:

```
┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────────┐    ┌─────────────┐
│ Preparación │───→│ Ingeniería   │───→│ Selección    │───→│ Optimización │───→│ Ensemble /  │
│ de datos    │    │ de features  │    │ de modelo    │    │ de hiperpar. │    │ Stacking    │
└─────────────┘    └──────────────┘    └─────────────┘    └──────────────┘    └─────────────┘
```

### ¿Qué automatiza cada librería?

| Librería | Feature Eng. | Model Selection | HPO | Ensemble | Velocidad | Facilidad |
|----------|:---:|:---:|:---:|:---:|:---:|:---:|
| **Auto-sklearn** | ✅ | ✅ | ✅ (SMAC) | ✅ | ⭐⭐ | ⭐⭐⭐ |
| **TPOT** | ✅ | ✅ | ✅ (genético) | Pipeline | ⭐ | ⭐⭐⭐ |
| **FLAML** | ❌ | ✅ | ✅ (CFO) | ❌ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **H2O AutoML** | ✅ | ✅ | ✅ (random/grid) | ✅ (stacking) | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Optuna** | ❌ | Manual | ✅ (TPE) | Manual | ⭐⭐⭐ | ⭐⭐ |
| **PyCaret** | ✅ | ✅ | ✅ (varios) | ✅ (blending) | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **LazyPredict** | ❌ | ✅ | ❌ | ❌ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |

### Enfoques de búsqueda

1. **SMAC** (Auto-sklearn): Bayesiana con Random Forest como surrogate
2. **Programación Genética** (TPOT): Evoluciona pipelines completos
3. **CFO / BlendSearch** (FLAML): Cost-Frugal Optimization, búsqueda económica
4. **TPE** (Optuna): Tree-structured Parzen Estimators
5. **Grid/Random + Stacking** (H2O): Fuerza bruta + ensamble inteligente

## 2. FLAML — Fast and Lightweight AutoML (Microsoft)

### Características
- **Extremadamente rápido**: diseñado para encontrar buenos modelos en segundos
- **Cost-Frugal Optimization (CFO)**: prioriza configuraciones baratas de evaluar
- **BlendSearch**: combina búsqueda local y global
- **Soporte**: clasificación, regresión, NLP, time series
- **Zero-shot AutoML**: usa meta-learning para sugerir configuraciones iniciales

### Algoritmo CFO
1. Comienza con configuraciones **baratas** (pocos estimadores, datos pequeños)
2. Si es prometedor, **aumenta** complejidad gradualmente
3. Optimiza **costo/rendimiento**, no solo rendimiento

In [ ]:
# ============================================================
# FLAML: AutoML rápido y eficiente
# ============================================================

try:
    from flaml import AutoML
    
    print("=== FLAML: Fast and Lightweight AutoML ===\n")
    
    # --- Clasificación ---
    X, y = datasets['Breast Cancer']['X'], datasets['Breast Cancer']['y']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    automl_clf = AutoML()
    
    t0 = time.time()
    automl_clf.fit(
        X_train, y_train,
        task="classification",
        time_budget=30,           # Máximo 30 segundos
        metric="accuracy",
        estimator_list=["lgbm", "rf", "xgboost", "extra_tree", "lrl1"],
        log_file_name="flaml_clf.log",
        verbose=0
    )
    t_clf = time.time() - t0
    
    y_pred = automl_clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print("--- Clasificación: Breast Cancer ---")
    print(f"  Mejor modelo: {automl_clf.best_estimator}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Tiempo: {t_clf:.1f}s")
    print(f"  Configuración: {automl_clf.best_config}")
    
    # --- Regresión ---
    ds_reg = [d for d in datasets.values() if d['tipo'] == 'reg'][0]
    X_r, y_r = ds_reg['X'], ds_reg['y']
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
    
    automl_reg = AutoML()
    
    t0 = time.time()
    automl_reg.fit(
        X_train_r, y_train_r,
        task="regression",
        time_budget=30,
        metric="r2",
        verbose=0
    )
    t_reg = time.time() - t0
    
    y_pred_r = automl_reg.predict(X_test_r)
    r2 = r2_score(y_test_r, y_pred_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
    
    print(f"\n--- Regresión ---")
    print(f"  Mejor modelo: {automl_reg.best_estimator}")
    print(f"  R²: {r2:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  Tiempo: {t_reg:.1f}s")
    
    # --- Explorar historial de FLAML ---
    print(f"\n--- Historial de búsqueda ---")
    print(f"  Modelos evaluados: {len(automl_clf.config_history)}")
    print(f"  Mejor loss en validación: {automl_clf.best_loss:.4f}")
    
    # Visualizar convergencia
    if hasattr(automl_clf, '_track_iter'):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Simular convergencia
        ax = axes[0]
        configs = list(automl_clf.config_history.items())
        losses = []
        best_so_far = float('inf')
        for _, (_, config) in enumerate(configs):
            try:
                loss = automl_clf.best_loss
                best_so_far = min(best_so_far, loss)
            except:
                pass
            losses.append(best_so_far)
        if losses:
            ax.plot(losses, 'b-', lw=2)
            ax.set_xlabel('Iteración')
            ax.set_ylabel('Mejor loss')
            ax.set_title('Convergencia de FLAML', fontweight='bold')
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    print("\n✅ FLAML: rápido, eficiente, excelente para prototipos")
    
except ImportError:
    print("⚠️  FLAML no instalado. Ejecutar: pip install flaml[automl]")
except Exception as e:
    print(f"⚠️  Error: {e}")

## 3. Optuna — Framework de optimización flexible

### Características
- **Define-by-run**: define el espacio de búsqueda en código Python normal
- **TPE** (Tree-structured Parzen Estimators): sampler bayesiano por defecto
- **Pruning**: detiene trials poco prometedores temprano (MedianPruner, HyperbandPruner)
- **Multi-objetivo**: puede optimizar múltiples métricas simultáneamente
- **Framework-agnostic**: funciona con sklearn, PyTorch, TensorFlow, XGBoost, etc.
- **Dashboard**: visualización web con Optuna Dashboard

### Algoritmo TPE
1. Divide las observaciones en "buenas" ($l(x)$) y "malas" ($g(x)$) según un percentil
2. Modela cada grupo con un KDE (Kernel Density Estimation)
3. Muestrea puntos que maximizan $l(x) / g(x)$

In [ ]:
# ============================================================
# Optuna: Optimización bayesiana flexible
# ============================================================

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("=== Optuna: Framework de Optimización ===\n")

X, y = datasets['Wine (3 clases)']['X'], datasets['Wine (3 clases)']['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Ejemplo 1: Optimizar un solo modelo ---
def objective_rf(trial):
    """Optimizar Random Forest."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }
    clf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    score = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy').mean()
    return score

study_rf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=False)

print("--- Optimización de Random Forest ---")
print(f"  Mejor accuracy: {study_rf.best_value:.4f}")
print(f"  Mejores hiperparámetros: {study_rf.best_params}")

# Evaluar en test
best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42)
best_rf.fit(X_train, y_train)
print(f"  Test accuracy: {best_rf.score(X_test, y_test):.4f}")

# --- Ejemplo 2: Selección automática de modelo ---
def objective_automl(trial):
    """Optuna como AutoML: seleccionar modelo + hiperparámetros."""
    modelo_nombre = trial.suggest_categorical('modelo', ['rf', 'gb', 'svm', 'knn', 'lr'])
    
    if modelo_nombre == 'rf':
        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int('rf_n_estimators', 50, 300),
            max_depth=trial.suggest_int('rf_max_depth', 3, 15),
            random_state=42
        )
    elif modelo_nombre == 'gb':
        clf = GradientBoostingClassifier(
            n_estimators=trial.suggest_int('gb_n_estimators', 50, 300),
            max_depth=trial.suggest_int('gb_max_depth', 3, 10),
            learning_rate=trial.suggest_float('gb_lr', 0.01, 0.3, log=True),
            random_state=42
        )
    elif modelo_nombre == 'svm':
        clf = SVC(
            C=trial.suggest_float('svm_C', 0.1, 100, log=True),
            kernel=trial.suggest_categorical('svm_kernel', ['rbf', 'linear', 'poly']),
            random_state=42
        )
    elif modelo_nombre == 'knn':
        clf = KNeighborsClassifier(
            n_neighbors=trial.suggest_int('knn_k', 1, 30),
            weights=trial.suggest_categorical('knn_weights', ['uniform', 'distance']),
        )
    else:
        clf = LogisticRegression(
            C=trial.suggest_float('lr_C', 0.01, 100, log=True),
            max_iter=500, random_state=42
        )
    
    score = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy').mean()
    return score

print("\n--- AutoML con Optuna (selección de modelo + HPO) ---")
study_auto = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_auto.optimize(objective_automl, n_trials=100, show_progress_bar=False)

print(f"  Mejor modelo: {study_auto.best_params.get('modelo')}")
print(f"  Mejor accuracy CV: {study_auto.best_value:.4f}")
print(f"  Parámetros: {study_auto.best_params}")

# --- Ejemplo 3: Multi-objetivo ---
def objective_multi(trial):
    """Optimizar accuracy Y velocidad simultáneamente."""
    n_est = trial.suggest_int('n_estimators', 10, 500)
    max_d = trial.suggest_int('max_depth', 2, 20)
    
    clf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42)
    
    t0 = time.time()
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='accuracy').mean()
    t_train = time.time() - t0
    
    return score, t_train  # Maximizar accuracy, minimizar tiempo

print("\n--- Optimización Multi-Objetivo ---")
study_multi = optuna.create_study(
    directions=['maximize', 'minimize'],
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_multi.optimize(objective_multi, n_trials=50, show_progress_bar=False)

# Pareto front
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Historia de optimización
ax = axes[0]
trials = study_auto.trials
accs = [t.value for t in trials if t.value is not None]
best_so_far = np.maximum.accumulate(accs)
ax.plot(accs, 'b.', alpha=0.3, markersize=5, label='Cada trial')
ax.plot(best_so_far, 'r-', lw=2, label='Mejor hasta ahora')
ax.set_xlabel('Trial'); ax.set_ylabel('Accuracy (CV)')
ax.set_title('Convergencia de Optuna', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# 2. Distribución por modelo
ax = axes[1]
modelos_trials = {}
for t in trials:
    if t.value is not None:
        m = t.params.get('modelo', 'unknown')
        modelos_trials.setdefault(m, []).append(t.value)

bp = ax.boxplot([modelos_trials.get(m, []) for m in ['rf', 'gb', 'svm', 'knn', 'lr']],
                labels=['RF', 'GB', 'SVM', 'KNN', 'LR'], patch_artist=True)
colores_bp = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for patch, color in zip(bp['boxes'], colores_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy por Modelo', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 3. Pareto front (multi-objetivo)
ax = axes[2]
accs_m = [t.values[0] for t in study_multi.trials if t.values is not None]
times_m = [t.values[1] for t in study_multi.trials if t.values is not None]
ax.scatter(times_m, accs_m, c='steelblue', alpha=0.6, s=40)

# Pareto front
pareto_trials = study_multi.best_trials
if pareto_trials:
    p_acc = [t.values[0] for t in pareto_trials]
    p_time = [t.values[1] for t in pareto_trials]
    idx_sort = np.argsort(p_time)
    ax.plot(np.array(p_time)[idx_sort], np.array(p_acc)[idx_sort], 
            'r-o', lw=2, markersize=8, label='Frente de Pareto')

ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('Accuracy')
ax.set_title('Multi-Objetivo: Accuracy vs Velocidad', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("\n✅ Optuna: máxima flexibilidad, ideal para investigación y producción")

## 4. TPOT — Pipelines evolucionados con programación genética

### Características
- **Programación genética**: evoluciona **pipelines completos** (preprocesamiento + modelo)
- Explora combinaciones de: escaladores, selección de features, PCA, modelos, etc.
- **Exporta código Python**: genera el script del mejor pipeline encontrado
- Basado en scikit-learn

### Cómo funciona
```
Generación 0:        Generación 1:        ...   Generación N:
[Scaler→PCA→RF]      [Scaler→PCA→RF]            [RobustScaler→
[Norm→SVM]      ───→  [Scaler→LDA→GB]    ───→    SelectKBest→
[PCA→GB]              [Norm→XGB]                  XGBoost]
[Raw→KNN]             [PCA→RF]                    (mejor pipeline)
```

Cada generación: **selección** → **cruce** → **mutación** → **evaluación**

In [ ]:
# ============================================================
# TPOT: Genetic Programming para pipelines de ML
# ============================================================

try:
    from tpot import TPOTClassifier, TPOTRegressor
    
    print("=== TPOT: Evolución de Pipelines ===\n")
    
    X, y = datasets['Breast Cancer']['X'], datasets['Breast Cancer']['y']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Configuración rápida para demo (en producción usar más generaciones)
    tpot_clf = TPOTClassifier(
        generations=5,           # Número de generaciones
        population_size=20,      # Tamaño de población
        cv=5,                    # Cross-validation
        scoring='accuracy',
        max_time_mins=2,         # Límite de tiempo en minutos
        max_eval_time_mins=0.5,  # Máximo por pipeline individual
        verbosity=2,
        random_state=42,
        n_jobs=-1
    )
    
    t0 = time.time()
    tpot_clf.fit(X_train, y_train)
    t_tpot = time.time() - t0
    
    acc_tpot = tpot_clf.score(X_test, y_test)
    
    print(f"\n--- Resultados ---")
    print(f"  Test accuracy: {acc_tpot:.4f}")
    print(f"  Tiempo: {t_tpot:.1f}s")
    print(f"  Pipelines evaluados: {len(tpot_clf.evaluated_individuals_)}")
    
    # Exportar el mejor pipeline como código Python
    tpot_clf.export('mejor_pipeline_tpot.py')
    print(f"\n--- Mejor Pipeline (código exportado) ---")
    with open('mejor_pipeline_tpot.py', 'r') as f:
        print(f.read())
    
    # Top 5 pipelines
    print("\n--- Top pipelines evaluados ---")
    sorted_pipes = sorted(tpot_clf.evaluated_individuals_.items(), 
                          key=lambda x: x[1].get('internal_cv_score', 0), reverse=True)
    for i, (pipe, info) in enumerate(sorted_pipes[:5]):
        score = info.get('internal_cv_score', 'N/A')
        # Acortar nombre del pipeline
        pipe_short = pipe[:80] + '...' if len(pipe) > 80 else pipe
        print(f"  {i+1}. CV={score:.4f} | {pipe_short}")

except ImportError:
    print("⚠️  TPOT no instalado. Ejecutar: pip install tpot")
except Exception as e:
    print(f"⚠️  Error con TPOT: {e}")
    print("   TPOT requiere scikit-learn compatible. Continuar con las demás librerías.")

## 5. LazyPredict — Benchmark rápido de modelos

### Características
- **Sin configuración**: evalúa decenas de modelos en una línea de código
- **Rápido**: usa parámetros por defecto (sin HPO)
- **Comparativo**: tabla ordenada de todos los modelos

### ¿Cuándo usarlo?
- **Exploración inicial**: ver qué familia de modelos funciona mejor
- **Baseline rápido**: tener un punto de referencia en segundos
- **NO para producción**: los parámetros no están optimizados

In [ ]:
# ============================================================
# LazyPredict: Benchmark rápido de múltiples modelos
# ============================================================

try:
    from lazypredict.Supervised import LazyClassifier, LazyRegressor
    
    print("=== LazyPredict: Benchmark de Modelos ===\n")
    
    # --- Clasificación ---
    X, y = datasets['Breast Cancer']['X'], datasets['Breast Cancer']['y']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print("--- Clasificación: Breast Cancer ---\n")
    lazy_clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models_clf, predictions_clf = lazy_clf.fit(X_train, X_test, y_train, y_test)
    
    print(models_clf.head(15).to_string())
    
    # --- Regresión ---
    ds_reg = [d for d in datasets.values() if d['tipo'] == 'reg'][0]
    X_r, y_r = ds_reg['X'], ds_reg['y']
    X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
    
    print("\n\n--- Regresión ---\n")
    lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
    models_reg, predictions_reg = lazy_reg.fit(X_tr_r, X_te_r, y_tr_r, y_te_r)
    
    print(models_reg.head(15).to_string())
    
    # Visualizar top modelos
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    ax = axes[0]
    top_clf = models_clf.head(10)
    colores = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_clf)))
    ax.barh(range(len(top_clf)), top_clf['Accuracy'], color=colores, edgecolor='black')
    ax.set_yticks(range(len(top_clf)))
    ax.set_yticklabels(top_clf.index, fontsize=9)
    ax.set_xlabel('Accuracy')
    ax.set_title('Top 10 Clasificadores (LazyPredict)', fontweight='bold')
    ax.invert_yaxis()
    
    ax = axes[1]
    top_reg = models_reg.head(10)
    colores = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_reg)))
    ax.barh(range(len(top_reg)), top_reg['R-Squared'], color=colores, edgecolor='black')
    ax.set_yticks(range(len(top_reg)))
    ax.set_yticklabels(top_reg.index, fontsize=9)
    ax.set_xlabel('R-Squared')
    ax.set_title('Top 10 Regresores (LazyPredict)', fontweight='bold')
    ax.invert_yaxis()
    
    plt.tight_layout(); plt.show()
    print("\n✅ LazyPredict: perfecto para exploración inicial rápida")
    
except ImportError:
    print("⚠️  LazyPredict no instalado. Ejecutar: pip install lazypredict")
except Exception as e:
    print(f"⚠️  Error: {e}")

## 6. PyCaret — AutoML Low-Code

### Características
- **Interfaz minimalista**: pipeline completo en ~5 líneas
- **Automatiza todo**: preprocesamiento, feature engineering, model selection, tuning, ensembles
- **Visualizaciones integradas**: curvas ROC, matrices de confusión, importancia de features
- **Deployment**: exportar modelo listo para producción

### Módulos
- `pycaret.classification`: clasificación
- `pycaret.regression`: regresión
- `pycaret.clustering`: clustering
- `pycaret.anomaly`: detección de anomalías
- `pycaret.time_series`: series temporales

In [ ]:
# ============================================================
# PyCaret: AutoML Low-Code
# ============================================================

try:
    from pycaret.classification import setup, compare_models, tune_model, blend_models
    from pycaret.classification import evaluate_model, predict_model, get_config
    from pycaret.classification import plot_model, pull
    
    print("=== PyCaret: AutoML Low-Code ===\n")
    
    # Preparar datos como DataFrame
    bc = load_breast_cancer()
    df = pd.DataFrame(bc.data, columns=bc.feature_names)
    df['target'] = bc.target
    
    # Setup: preprocesamiento automático
    print("--- Paso 1: Setup (preprocesamiento automático) ---\n")
    s = setup(
        data=df,
        target='target',
        session_id=42,
        verbose=False,
        html=False,
        train_size=0.8,
        fold=5,
        normalize=True,
        remove_multicollinearity=True,
        multicollinearity_threshold=0.95
    )
    
    # Comparar modelos
    print("--- Paso 2: Comparar modelos ---\n")
    best_models = compare_models(n_select=5, sort='Accuracy')
    
    # Mostrar resultados
    results = pull()
    print(results.head(10).to_string())
    
    # Tunear el mejor modelo
    print("\n--- Paso 3: Tune del mejor modelo ---\n")
    tuned = tune_model(best_models[0], optimize='Accuracy', n_iter=20)
    tuned_results = pull()
    print(f"  Modelo tuneado: {type(tuned).__name__}")
    print(f"  Accuracy: {tuned_results['Accuracy'].mean():.4f}")
    
    # Ensemble: blending
    print("\n--- Paso 4: Blending (ensemble) ---\n")
    try:
        blended = blend_models(best_models[:3])
        blend_results = pull()
        print(f"  Blend accuracy: {blend_results['Accuracy'].mean():.4f}")
    except:
        print("  Blending no disponible en esta versión")
    
    # Predicción en test
    print("\n--- Paso 5: Predicción en test set ---\n")
    predictions = predict_model(tuned)
    print(predictions[['target', 'prediction_label', 'prediction_score']].head(10).to_string())
    
    print("\n✅ PyCaret: el camino más corto de datos a modelo")
    
except ImportError:
    print("⚠️  PyCaret no instalado. Ejecutar: pip install pycaret")
    print("   PyCaret es pesado (~1GB). Instalar solo cuando se necesite.")
except Exception as e:
    print(f"⚠️  Error con PyCaret: {e}")

## 7. H2O AutoML — Escalable y con Stacking

### Características
- **Escalable**: diseñado para datasets grandes (distribuido)
- **Stacking automático**: crea ensembles apilados de los mejores modelos
- **Algoritmos**: GBM, XGBoost, Deep Learning, GLM, DRF, Stacked Ensembles
- **Explicabilidad**: SHAP values integrados
- **Leaderboard**: ranking automático de todos los modelos entrenados

### Arquitectura
H2O corre como un **servidor Java** separado. Python se comunica via API.

In [ ]:
# ============================================================
# H2O AutoML: Escalable y con Stacking automático
# ============================================================

try:
    import h2o
    from h2o.automl import H2OAutoML
    
    print("=== H2O AutoML ===\n")
    
    # Iniciar servidor H2O
    h2o.init(nthreads=-1, max_mem_size="2G", verbose=False)
    
    # Preparar datos
    bc = load_breast_cancer()
    df = pd.DataFrame(bc.data, columns=bc.feature_names)
    df['target'] = bc.target.astype(str)  # H2O necesita string para clasificación
    
    # Convertir a H2OFrame
    hf = h2o.H2OFrame(df)
    hf['target'] = hf['target'].asfactor()
    
    # Split
    train, test = hf.split_frame(ratios=[0.8], seed=42)
    
    # AutoML
    aml = H2OAutoML(
        max_models=20,
        max_runtime_secs=120,
        seed=42,
        sort_metric='accuracy',
        exclude_algos=['DeepLearning']  # Excluir DL para velocidad
    )
    
    aml.train(y='target', training_frame=train)
    
    # Leaderboard
    print("--- Leaderboard ---")
    lb = aml.leaderboard.as_data_frame()
    print(lb.head(10).to_string())
    
    # Mejor modelo
    print(f"\n--- Mejor modelo: {aml.leader.model_id} ---")
    perf = aml.leader.model_performance(test)
    print(f"  Accuracy: {perf.accuracy()[0][1]:.4f}")
    print(f"  AUC: {perf.auc():.4f}")
    print(f"  F1: {perf.F1()[0][1]:.4f}")
    
    # Importancia de variables
    if hasattr(aml.leader, 'varimp'):
        varimp = aml.leader.varimp(use_pandas=True)
        if varimp is not None:
            print(f"\n--- Top 10 features más importantes ---")
            print(varimp.head(10).to_string())
    
    # Stacked Ensemble
    se_models = [m for m in lb['model_id'] if 'StackedEnsemble' in m]
    if se_models:
        print(f"\n--- Stacked Ensembles generados: {len(se_models)} ---")
        for m in se_models:
            print(f"  {m}")
    
    h2o.shutdown(prompt=False)
    print("\n✅ H2O: ideal para producción y datasets grandes")

except ImportError:
    print("⚠️  H2O no instalado. Ejecutar: pip install h2o")
    print("   Requiere Java Runtime Environment (JRE)")
except Exception as e:
    print(f"⚠️  Error con H2O: {e}")
    try:
        h2o.shutdown(prompt=False)
    except:
        pass

## 8. Auto-sklearn — AutoML con Meta-Learning

### Características
- **Meta-Learning**: usa experiencia de datasets anteriores para warmstart
- **Basado en SMAC**: Sequential Model-Based Algorithm Configuration
- **Ensembles automáticos**: combina los mejores modelos con selection ensemble
- **Espacio de búsqueda**: 15 clasificadores, 14 preprocesadores, ~110 hiperparámetros

### Pipeline

```
                Meta-Learning
                     │
                     ▼
Dataset ──→ Warm Start Configs ──→ SMAC (Bayesiana) ──→ Ensemble Selection
                                        │                      │
                                   Evaluar pipeline       Combinar mejores
                                   (modelo + preproceso)   modelos
```

> ⚠️ **Nota**: auto-sklearn solo funciona en Linux. En macOS/Windows usar WSL o Docker.

In [ ]:
# ============================================================
# Auto-sklearn: AutoML con Meta-Learning
# ============================================================

try:
    import autosklearn.classification
    import autosklearn.regression
    
    print("=== Auto-sklearn: AutoML con Meta-Learning ===\n")
    
    X, y = datasets['Breast Cancer']['X'], datasets['Breast Cancer']['y']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    automl = autosklearn.classification.AutoSklearnClassifier(
        time_left_for_this_task=120,  # 2 minutos total
        per_run_time_limit=30,        # 30s por modelo
        n_jobs=-1,
        seed=42,
        memory_limit=4096,
        ensemble_size=10,
        initial_configurations_via_metalearning=25,
    )
    
    t0 = time.time()
    automl.fit(X_train, y_train)
    t_auto = time.time() - t0
    
    y_pred = automl.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"--- Resultados ---")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Tiempo: {t_auto:.1f}s")
    
    # Modelos en el ensemble
    print(f"\n--- Ensemble final ---")
    print(automl.show_models())
    
    # Estadísticas
    print(f"\n--- Estadísticas ---")
    print(automl.sprint_statistics())
    
    print("\n✅ Auto-sklearn: el más completo para scikit-learn")
    
except ImportError:
    print("⚠️  Auto-sklearn no instalado o no disponible en este OS")
    print("   Solo funciona en Linux. Ejecutar: pip install auto-sklearn")
    print("   En macOS/Windows: usar Docker o WSL")
except Exception as e:
    print(f"⚠️  Error: {e}")

## 9. Comparación Completa: Benchmark Multi-Dataset

Ahora comparamos todas las librerías disponibles en múltiples datasets para tener una vista global de rendimiento vs tiempo.

In [ ]:
# ============================================================
# Benchmark comparativo: AutoML vs Baselines
# ============================================================

print("=" * 70)
print("  BENCHMARK: AutoML vs Baselines en múltiples datasets")
print("=" * 70)

def baseline_manual(X_train, y_train, X_test, y_test, tipo='clf'):
    """Baseline: mejor modelo con GridSearch manual."""
    if tipo == 'clf':
        modelos = {
            'RF': RandomForestClassifier(n_estimators=200, random_state=42),
            'GB': GradientBoostingClassifier(n_estimators=200, random_state=42),
            'LR': LogisticRegression(max_iter=500, random_state=42),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'SVM': SVC(random_state=42),
        }
        metric = 'accuracy'
    else:
        modelos = {
            'RF': RandomForestRegressor(n_estimators=200, random_state=42),
            'GB': GradientBoostingRegressor(n_estimators=200, random_state=42),
            'Ridge': Ridge(),
        }
        metric = 'r2'
    
    mejor_nombre, mejor_score = None, -np.inf
    for nombre, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        if tipo == 'clf':
            score = accuracy_score(y_test, modelo.predict(X_test))
        else:
            score = r2_score(y_test, modelo.predict(X_test))
        if score > mejor_score:
            mejor_score = score
            mejor_nombre = nombre
    
    return mejor_nombre, mejor_score

resultados_benchmark = {}

datasets_clf = {k: v for k, v in datasets.items() if v['tipo'] == 'clf'}

for ds_nombre, ds_info in datasets_clf.items():
    print(f"\n--- {ds_nombre} ({ds_info['n_samples']} samples, {ds_info['n_features']} features) ---")
    
    X, y = ds_info['X'], ds_info['y']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    resultados_ds = {}
    
    # 1. Baseline manual
    t0 = time.time()
    nombre_bl, acc_bl = baseline_manual(X_train, y_train, X_test, y_test, 'clf')
    t_bl = time.time() - t0
    resultados_ds['Baseline (manual)'] = {'score': acc_bl, 'time': t_bl, 'modelo': nombre_bl}
    
    # 2. FLAML
    try:
        from flaml import AutoML as FLAML_AutoML
        automl_f = FLAML_AutoML()
        t0 = time.time()
        automl_f.fit(X_train, y_train, task="classification", time_budget=15,
                     metric="accuracy", verbose=0)
        t_f = time.time() - t0
        acc_f = accuracy_score(y_test, automl_f.predict(X_test))
        resultados_ds['FLAML'] = {'score': acc_f, 'time': t_f, 'modelo': automl_f.best_estimator}
    except:
        pass
    
    # 3. Optuna
    try:
        def obj_optuna(trial):
            m = trial.suggest_categorical('m', ['rf', 'gb', 'knn', 'lr'])
            if m == 'rf':
                clf = RandomForestClassifier(
                    n_estimators=trial.suggest_int('n', 50, 300),
                    max_depth=trial.suggest_int('d', 3, 15), random_state=42)
            elif m == 'gb':
                clf = GradientBoostingClassifier(
                    n_estimators=trial.suggest_int('n2', 50, 300),
                    learning_rate=trial.suggest_float('lr2', 0.01, 0.3, log=True), random_state=42)
            elif m == 'knn':
                clf = KNeighborsClassifier(n_neighbors=trial.suggest_int('k', 1, 20))
            else:
                clf = LogisticRegression(C=trial.suggest_float('C', 0.01, 100, log=True),
                                         max_iter=500, random_state=42)
            return cross_val_score(clf, X_train, y_train, cv=3, scoring='accuracy').mean()
        
        study_b = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
        t0 = time.time()
        study_b.optimize(obj_optuna, n_trials=50, show_progress_bar=False)
        t_o = time.time() - t0
        
        # Reconstruir mejor modelo
        bp = study_b.best_params
        if bp['m'] == 'rf':
            best_clf = RandomForestClassifier(n_estimators=bp['n'], max_depth=bp['d'], random_state=42)
        elif bp['m'] == 'gb':
            best_clf = GradientBoostingClassifier(n_estimators=bp['n2'], learning_rate=bp['lr2'], random_state=42)
        elif bp['m'] == 'knn':
            best_clf = KNeighborsClassifier(n_neighbors=bp['k'])
        else:
            best_clf = LogisticRegression(C=bp['C'], max_iter=500, random_state=42)
        best_clf.fit(X_train, y_train)
        acc_o = accuracy_score(y_test, best_clf.predict(X_test))
        resultados_ds['Optuna'] = {'score': acc_o, 'time': t_o, 'modelo': bp['m']}
    except:
        pass
    
    resultados_benchmark[ds_nombre] = resultados_ds
    
    # Imprimir resultados del dataset
    for metodo, info in resultados_ds.items():
        bar = '█' * int(info['score'] * 40)
        print(f"  {metodo:20s}: {info['score']:.4f} ({info['time']:5.1f}s) [{info['modelo']}]  {bar}")

# --- Visualización final ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Accuracy por método y dataset
ax = axes[0]
metodos_all = set()
for ds_res in resultados_benchmark.values():
    metodos_all.update(ds_res.keys())
metodos_all = sorted(metodos_all)

x = np.arange(len(resultados_benchmark))
width = 0.25
colores_metodos = {'Baseline (manual)': '#e74c3c', 'FLAML': '#2ecc71', 'Optuna': '#3498db'}

for i, metodo in enumerate(metodos_all):
    scores = []
    for ds_nombre in resultados_benchmark:
        info = resultados_benchmark[ds_nombre].get(metodo, {'score': 0})
        scores.append(info['score'])
    
    color = colores_metodos.get(metodo, f'C{i}')
    ax.bar(x + i*width - width, scores, width, label=metodo, color=color, 
           alpha=0.8, edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(list(resultados_benchmark.keys()), fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Comparación: AutoML vs Baseline', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0.7, 1.05)

# 2. Tiempo vs Accuracy (scatter)
ax = axes[1]
for metodo in metodos_all:
    times_m, scores_m = [], []
    for ds_nombre in resultados_benchmark:
        info = resultados_benchmark[ds_nombre].get(metodo)
        if info:
            times_m.append(info['time'])
            scores_m.append(info['score'])
    
    color = colores_metodos.get(metodo, 'gray')
    ax.scatter(times_m, scores_m, c=color, s=100, alpha=0.7, label=metodo, 
               edgecolors='black', zorder=3)

ax.set_xlabel('Tiempo (segundos)')
ax.set_ylabel('Accuracy')
ax.set_title('Trade-off: Accuracy vs Tiempo', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 10. Guía Práctica: ¿Cuál AutoML usar?

### Árbol de decisión

```
¿Necesitas AutoML?
│
├── Exploración rápida (< 1 min)
│   ├── LazyPredict → ver qué modelo funciona
│   └── FLAML (time_budget=30) → modelo rápido
│
├── Prototipo (1-10 min)
│   ├── PyCaret → interfaz más fácil, reportes bonitos
│   ├── FLAML → rápido y eficiente
│   └── Optuna → si necesitas control total
│
├── Producción (10+ min)
│   ├── H2O AutoML → escalable, stacking, explicabilidad
│   ├── Auto-sklearn → meta-learning, ensembles robustos
│   └── Optuna → máxima flexibilidad
│
└── Pipeline completo (preprocesamiento incluido)
    ├── TPOT → programación genética, exporta código
    ├── PyCaret → todo automático
    └── Auto-sklearn → pipeline completo
```

### Tabla resumen

| Criterio | Mejor opción |
|---------|-------------|
| **Más rápido** | FLAML, LazyPredict |
| **Más fácil** | PyCaret, LazyPredict |
| **Más flexible** | Optuna |
| **Mejor ensemble** | H2O, Auto-sklearn |
| **Pipeline completo** | TPOT, PyCaret |
| **Escalabilidad** | H2O |
| **Meta-Learning** | Auto-sklearn |
| **Multi-objetivo** | Optuna |
| **Exportar código** | TPOT |
| **Deep Learning** | Optuna + PyTorch |

### Lo que AutoML NO hace
- ❌ Entender el problema de negocio
- ❌ Limpiar datos mal etiquetados
- ❌ Definir la métrica correcta
- ❌ Feature engineering experto (dominio específico)
- ❌ Garantizar fairness/sesgo
- ❌ Monitorear drift en producción

**AutoML es una herramienta, no un reemplazo del data scientist.**

In [ ]:
# ============================================================
# Resumen visual final
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.axis('off')

# Datos de librerías
librerias = [
    ('FLAML', 2, 8.5, 'Microsoft', '#2ecc71', 'Rapido, eficiente\nCFO/BlendSearch'),
    ('Optuna', 5, 8.5, 'Preferred Networks', '#3498db', 'Flexible, TPE\nMulti-objetivo'),
    ('LazyPredict', 8, 8.5, 'Open Source', '#f39c12', 'Benchmark rapido\nSin configuracion'),
    ('TPOT', 2, 5.5, 'U. Pennsylvania', '#e74c3c', 'Prog. genetica\nExporta codigo'),
    ('PyCaret', 5, 5.5, 'Open Source', '#9b59b6', 'Low-code, todo\nen 5 lineas'),
    ('H2O', 8, 5.5, 'H2O.ai', '#1abc9c', 'Escalable, stacking\nProduccion'),
    ('Auto-sklearn', 3.5, 2.5, 'U. Freiburg', '#e67e22', 'Meta-learning\nEnsembles SMAC'),
]

for nombre, x, y, org, color, desc in librerias:
    # Caja
    rect = plt.Rectangle((x-1.2, y-0.9), 2.4, 1.8, 
                          facecolor=color, alpha=0.2, edgecolor=color, lw=2,
                          transform=ax.transData, zorder=2)
    ax.add_patch(rect)
    
    # Texto
    ax.text(x, y+0.5, nombre, ha='center', va='center', fontsize=13, 
            fontweight='bold', color=color)
    ax.text(x, y-0.05, org, ha='center', va='center', fontsize=8, 
            color='gray', style='italic')
    ax.text(x, y-0.5, desc, ha='center', va='center', fontsize=8)

ax.text(5, 10, 'Ecosistema de AutoML', ha='center', va='center', 
        fontsize=18, fontweight='bold')
ax.text(5, 9.5, 'Cada libreria tiene su nicho - elige segun tu necesidad', 
        ha='center', va='center', fontsize=11, color='gray')

plt.tight_layout()
plt.savefig('automl_ecosistema.png', dpi=100, bbox_inches='tight')
plt.show()

print("=" * 60)
print("  RESUMEN: AutoML")
print("=" * 60)
print()
print("  FLAML     → El mas rapido, ideal para empezar")
print("  Optuna    → El mas flexible, ideal para investigacion")
print("  TPOT      → Pipelines completos evolucionados")
print("  PyCaret   → El mas facil, ideal para prototipos")
print("  H2O       → El mas escalable, ideal para produccion")
print("  Auto-sk   → El mas completo, meta-learning + ensembles")
print("  LazyPred  → El mas rapido para benchmark inicial")
print()
print("  AutoML NO reemplaza al data scientist.")
print("  Es una herramienta que AMPLIFICA su productividad.")